<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part D: Deep Learning Approaches</h2>
<h2>Notebook D01: Neural Networks for Time Series</h2>
</div>

Part C treated forecasting as supervised learning on a feature matrix. Part D keeps that framing and
changes the model: instead of trees or linear coefficients, a network of weighted connections trained by
gradient descent.

We start with the simplest such model, the **multilayer perceptron**, on exactly the features and the
split used in Notebooks C02 and C03, so the comparison is direct. Along the way the notebook covers the
things neural networks demand that trees did not: scaled inputs, a training loop you write yourself, a
stopping rule, and an honest account of how much the random seed moves the answer.

> This notebook needs PyTorch, which is not in the default install. Run `uv sync --group dl` first.

---

**Contents**

1. [Imports and Data](#1.-Imports-and-Data)
2. [Why a Neural Network](#2.-Why-a-Neural-Network)
3. [Preparing Data for a Network](#3.-Preparing-Data-for-a-Network)
4. [Building and Training an MLP](#4.-Building-and-Training-an-MLP)
5. [Watching It Train](#5.-Watching-It-Train)
6. [How Much Does the Random Seed Matter?](#6.-How-Much-Does-the-Random-Seed-Matter?)
7. [The Comparison](#7.-The-Comparison)
8. [What an MLP Cannot Do](#8.-What-an-MLP-Cannot-Do)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-Imports-and-Data">1. Imports and Data</h3>
</div>

In [ ]:
import importlib.util
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import nb_config

sns.set_theme(style="whitegrid")

TORCH_AVAILABLE = importlib.util.find_spec("torch") is not None

if TORCH_AVAILABLE:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    print(f"PyTorch {torch.__version__}")
    print(f"Threads: {torch.get_num_threads()}")
else:
    print("PyTorch is not installed. Run 'uv sync --group dl' to follow this notebook.")

In [ ]:
sales = pd.read_csv(nb_config.ROSSMANN_TRAIN_PATH, parse_dates=["Date"], low_memory=False)
store = sales[sales["Store"] == 1].set_index("Date").sort_index().asfreq("D")
target = store["Sales"].astype(float)


def build_features(target, store):
    """The feature matrix from C01, unchanged."""
    features = pd.DataFrame(index=target.index)

    for lag in (1, 2, 7, 14, 28):
        features[f"lag_{lag}"] = target.shift(lag)

    history = target.shift(1)
    for window in (7, 28):
        features[f"roll_mean_{window}"] = history.rolling(window).mean()
        features[f"roll_std_{window}"] = history.rolling(window).std()

    features["day_of_week"] = target.index.dayofweek
    features["day_of_month"] = target.index.day
    features["month"] = target.index.month
    features["days_since_start"] = (target.index - target.index[0]).days

    for k in (1, 2):
        position = target.index.dayofyear / 365.25
        features[f"fourier_sin_{k}"] = np.sin(2 * np.pi * k * position)
        features[f"fourier_cos_{k}"] = np.cos(2 * np.pi * k * position)

    features["open"] = store["Open"]
    features["promo"] = store["Promo"]
    features["school_holiday"] = store["SchoolHoliday"]

    return features


features = build_features(target, store)
complete = features.notna().all(axis=1)
X, y = features[complete], target[complete]

TEST_DAYS, VALIDATION_DAYS = 90, 90
test_start = len(X) - TEST_DAYS
validation_start = test_start - VALIDATION_DAYS

print(f"{X.shape[1]} features")
print(f"Train {validation_start} days, validation {VALIDATION_DAYS}, test {TEST_DAYS}")

The validation period is doing real work in this notebook, not just providing a second opinion. A network
has no natural stopping point: it keeps improving on the training data indefinitely, and only held-out data
can say when to stop. That makes the middle period a requirement rather than a nicety.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-Why-a-Neural-Network">2. Why a Neural Network</h3>
</div>

A **multilayer perceptron** is a stack of layers. Each layer multiplies its input by a matrix of weights,
adds a bias, and applies a non-linear function. Stack two or three and the composition can approximate
essentially any smooth relationship between inputs and output.

The non-linearity is the whole point. Without it, a stack of matrix multiplications collapses into a
single matrix, and you have reinvented linear regression with extra steps. `ReLU`, which passes positive
values through and zeroes negatives, is the usual choice because it is cheap and trains well.

What this buys over Part C:

- **Interactions for free.** The network learns combinations of features without being told which to try.
- **It extrapolates.** Unlike the trees of Notebook [C02](./C02_Machine_learning_models.ipynb), a network's
  output is not bounded by the values it saw in training.
- **It scales.** The same architecture handles many series, many inputs, and multiple outputs at once.

What it costs:

- **Data.** Networks have many parameters and need enough examples to pin them down.
- **Care.** Scaling, learning rates, stopping rules and initialisation all matter, and all can go wrong
  quietly.
- **Determinism.** Two runs of the same code give different answers, which section 6 measures.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-Preparing-Data-for-a-Network">3. Preparing Data for a Network</h3>
</div>

Trees split on thresholds, so the scale of a feature is irrelevant to them. Networks multiply features by
weights and take gradients, so scale is everything. Our `lag_1` runs in the thousands while
`fourier_sin_1` sits between -1 and 1; with a shared learning rate, the gradients from those two columns
differ by three orders of magnitude.

Both the features **and the target** are standardised, and the scalers are fitted on the **training period
only**. Fitting a scaler on all the data is a quiet form of the leakage Notebook
[C01](./C01_Feature_engineering.ipynb) warned about: the mean and standard deviation would carry
information from the test period.

In [ ]:
def prepare_tensors(X, y, validation_start, test_start, scale=True):
    """Standardise on the training period only, and convert to tensors."""
    feature_scaler, target_scaler = StandardScaler(), StandardScaler()

    X_train_raw = X.iloc[:validation_start]
    y_train_raw = y.iloc[:validation_start].values.reshape(-1, 1)

    if scale:
        feature_scaler.fit(X_train_raw)
        target_scaler.fit(y_train_raw)
        transform_X = feature_scaler.transform
        transform_y = target_scaler.transform
    else:
        transform_X = lambda values: np.asarray(values, dtype=float)
        transform_y = lambda values: np.asarray(values, dtype=float)

    def to_tensor(values):
        return torch.tensor(np.asarray(values, dtype=np.float32))

    tensors = {
        "X_train": to_tensor(transform_X(X_train_raw)),
        "y_train": to_tensor(transform_y(y_train_raw)),
        "X_validation": to_tensor(transform_X(X.iloc[validation_start:test_start])),
        "X_test": to_tensor(transform_X(X.iloc[test_start:])),
    }

    def invert(predictions):
        predictions = np.asarray(predictions).reshape(-1, 1)
        return (target_scaler.inverse_transform(predictions) if scale else predictions).ravel()

    return tensors, invert


if TORCH_AVAILABLE:
    tensors, to_original_units = prepare_tensors(X, y, validation_start, test_start)

    print("Feature ranges before scaling:")
    print(X[["lag_1", "fourier_sin_1"]].agg(["min", "max"]).round(2).to_string())
    print()
    print("After scaling (training period):")
    scaled = pd.DataFrame(tensors["X_train"].numpy(), columns=X.columns)
    print(scaled[["lag_1", "fourier_sin_1"]].agg(["min", "max"]).round(2).to_string())

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-Building-and-Training-an-MLP">4. Building and Training an MLP</h3>
</div>

Now the model. `nn.Sequential` stacks layers; `Linear(a, b)` is the weight matrix; `ReLU` is the
non-linearity; `Dropout` randomly zeroes a fraction of activations during training, which discourages the
network from relying on any single path.

The training loop is written out rather than hidden behind a `.fit()`, because every line of it is a
decision you may need to change:

1. Shuffle the training rows into mini-batches. Shuffling **rows of an already-built feature matrix** is
   fine: each row is a self-contained prediction problem, and the temporal order is already encoded in the
   lag columns. This is not the same as shuffling a train/test split.
2. For each batch: forward pass, compute the loss, backpropagate, step the optimiser.
3. After each epoch, score the validation period and remember the best weights seen so far.

That last step is **early stopping**, and it is how a network knows when to quit.

In [ ]:
def build_network(n_features, hidden=(64, 32), dropout=0.1):
    """A small MLP: two hidden layers, ReLU activations, dropout for regularisation."""
    layers = []
    previous = n_features

    for width in hidden:
        layers += [nn.Linear(previous, width), nn.ReLU()]
        if dropout:
            layers.append(nn.Dropout(dropout))
        previous = width

    layers.append(nn.Linear(previous, 1))
    return nn.Sequential(*layers)


def train_network(tensors, invert, y_validation, seed=0, epochs=300, batch_size=32,
                  learning_rate=1e-3, weight_decay=1e-4):
    """Train with mini-batches, keeping the weights that score best on validation."""
    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)

    network = build_network(tensors["X_train"].shape[1])
    optimiser = torch.optim.Adam(
        network.parameters(), lr=learning_rate, weight_decay=weight_decay
    )
    loss_function = nn.MSELoss()

    loader = DataLoader(
        TensorDataset(tensors["X_train"], tensors["y_train"]),
        batch_size=batch_size, shuffle=True, generator=generator,
    )

    history = []
    best = {"validation_mae": np.inf, "epoch": 0, "weights": None}

    for epoch in range(epochs):
        network.train()
        for batch_X, batch_y in loader:
            optimiser.zero_grad()
            loss = loss_function(network(batch_X), batch_y)
            loss.backward()
            optimiser.step()

        network.eval()
        with torch.no_grad():
            training_loss = loss_function(network(tensors["X_train"]), tensors["y_train"]).item()
            validation_prediction = invert(network(tensors["X_validation"]).numpy())

        validation_mae = mean_absolute_error(y_validation, validation_prediction)
        history.append({"epoch": epoch, "training_loss": training_loss,
                        "validation_mae": validation_mae})

        if validation_mae < best["validation_mae"]:
            best = {
                "validation_mae": validation_mae, "epoch": epoch,
                "weights": {k: v.clone() for k, v in network.state_dict().items()},
            }

    network.load_state_dict(best["weights"])
    network.eval()

    return network, pd.DataFrame(history), best


y_validation = y.iloc[validation_start:test_start]
y_test = y.iloc[test_start:]

if TORCH_AVAILABLE:
    network, history, best = train_network(tensors, to_original_units, y_validation)

    with torch.no_grad():
        mlp_prediction = to_original_units(network(tensors["X_test"]).numpy())

    print(f"Best validation MAE {best['validation_mae']:.1f}, reached at epoch {best['epoch']}")
    print(f"Test MAE: {mean_absolute_error(y_test, mlp_prediction):.1f}")

Before comparing that against anything, it is worth seeing what happens without the scaling step, because
it is the single easiest way to waste an afternoon.

In [ ]:
if TORCH_AVAILABLE:
    unscaled_tensors, unscaled_invert = prepare_tensors(
        X, y, validation_start, test_start, scale=False
    )
    unscaled_network, unscaled_history, unscaled_best = train_network(
        unscaled_tensors, unscaled_invert, y_validation, epochs=100
    )
    scaled_history = train_network(tensors, to_original_units, y_validation, epochs=100)[1]

    with torch.no_grad():
        unscaled_prediction = unscaled_invert(unscaled_network(unscaled_tensors["X_test"]).numpy())

    print(f"Test MAE     with scaling: {mean_absolute_error(y_test, mlp_prediction):8.1f}")
    print(f"          without scaling: "
          f"{mean_absolute_error(y_test, unscaled_prediction):8.1f}")
    print()
    print("Validation MAE over 100 epochs:")
    for label, frame in [("scaled", scaled_history), ("unscaled", unscaled_history)]:
        print(f"  {label:<9} best {frame['validation_mae'].min():7.1f}   "
              f"final {frame['validation_mae'].iloc[-1]:7.1f}   "
              f"worst {frame['validation_mae'].max():7.1f}")

Unscaled is about 45% worse on the test set, and the validation numbers show the damage is systematic
rather than a single bad epoch. The unscaled run is worse on all three measures, and by a telling margin:
its **best** epoch, at 573, is worse than the scaled run's **final** epoch at 414.

Both runs wander, which is normal while a network is still finding its footing, and both end worse than
their best epoch. The difference is the level. The optimiser cannot find one learning rate that suits both
a feature in the thousands and one bounded by 1, so every epoch of the unscaled run is handicapped.

The interesting part is that the damage is not worse. Early stopping salvaged a mediocre model from a
broken setup by keeping the weights from a lucky epoch, and that is exactly what makes this failure hard
to spot: the code runs, a number comes out, and it is merely bad rather than obviously absurd. Reporting
the final epoch instead of the best would have been substantially worse again.

**Scale your inputs**, and read the training curve rather than only the final score.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-Watching-It-Train">5. Watching It Train</h3>
</div>

The training history is the main diagnostic a network offers. Plot the training loss against the
validation error and the two curves tell you what the model is doing.

In [ ]:
if TORCH_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    axes[0].plot(history["epoch"], history["training_loss"], color="steelblue", linewidth=1.2)
    axes[0].set_title("Training loss (scaled units)", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("MSE")
    axes[0].set_yscale("log")

    axes[1].plot(history["epoch"], history["validation_mae"], color="darkorange", linewidth=1.2)
    axes[1].axvline(best["epoch"], color="crimson", linestyle="--", linewidth=1.2,
                    label=f"best epoch ({best['epoch']})")
    axes[1].axhline(best["validation_mae"], color="crimson", linestyle=":", linewidth=1.0)
    axes[1].set_title("Validation MAE (original units)", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("MAE")
    axes[1].legend()

    for ax in axes:
        ax.grid(linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

The training loss falls steadily and keeps falling: given enough epochs it would approach zero, because the
network has enough capacity to memorise 800 rows. The validation error tells the real story. It drops
quickly, reaches a minimum, and then drifts sideways and upward as the network starts fitting noise.

The gap between those two curves **is** overfitting, and the point where the orange line bottoms out is
the only moment worth keeping. That is what the early stopping in `train_network` does: it holds on to the
weights from the best epoch rather than whatever the last epoch happened to produce.

Note that the best epoch is nowhere near the last one. Training for longer would have made the model
worse.

**Exercise.** Set `dropout=0.0` and `weight_decay=0.0` in `train_network` and plot the curves again. How much sooner does the validation error start rising, and what does that tell you about what those two settings were doing?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-How-Much-Does-the-Random-Seed-Matter?">6. How Much Does the Random Seed Matter?</h3>
</div>

Everything so far came from one run with `seed=0`. The random seed controls the initial weights, the
dropout mask, and the order the batches are shuffled in, and none of those are neutral.

A random forest with a fixed seed is reproducible, and changing the seed moves the result very little. A
network is a different proposition, and the honest way to report one is to train it several times.

> **This cell trains five networks and takes a minute or two.**

In [ ]:
if TORCH_AVAILABLE:
    seed_results = []

    for seed in range(5):
        seed_network, _, seed_best = train_network(
            tensors, to_original_units, y_validation, seed=seed
        )
        with torch.no_grad():
            prediction = to_original_units(seed_network(tensors["X_test"]).numpy())

        seed_results.append({
            "seed": seed,
            "best epoch": seed_best["epoch"],
            "validation MAE": seed_best["validation_mae"],
            "test MAE": mean_absolute_error(y_test, prediction),
        })

    seeds = pd.DataFrame(seed_results).set_index("seed")
    print(seeds.round(1).to_string())
    print()
    print(f"test MAE across seeds: mean {seeds['test MAE'].mean():.1f}, "
          f"std {seeds['test MAE'].std():.1f}, "
          f"range {seeds['test MAE'].min():.1f} to {seeds['test MAE'].max():.1f}")

Same data, same architecture, same code. The test MAE spans roughly 80 points between the luckiest and
unluckiest seed, and the epoch at which each run peaked ranges from the fifties to past two hundred.

Two consequences, and both are about honesty rather than technique:

**Reporting a single run is cherry-picking**, whether or not you meant it that way. If you train once, get
301, and publish that, you have reported the best of a distribution whose mean is nearer 327.

**Comparisons need the same treatment.** A network that beats a random forest by 20 MAE on one seed has
not beaten anything. Average over several runs, quote the spread, and treat differences smaller than that
spread as noise, exactly as Notebook [A06](./A06_Evaluating_models.ipynb) argued for forecast origins.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="7.-The-Comparison">7. The Comparison</h3>
</div>

Now place the network alongside the models from Part C, on the same features, the same split and the same
metric. The network is reported as its mean across the five seeds, which is the number section 6 says is
the honest one.

In [ ]:
ridge = make_pipeline(StandardScaler(), Ridge(alpha=10.0))
ridge.fit(X.iloc[:validation_start], y.iloc[:validation_start])
ridge_mae = mean_absolute_error(y_test, ridge.predict(X.iloc[test_start:]))

comparison = pd.DataFrame([
    {"Model": "Random forest (C02)", "MAE": 232.2, "Source": "Notebook C02"},
    {"Model": "LightGBM (C02)", "MAE": 240.0, "Source": "Notebook C02"},
    {"Model": "MLP (mean of 5 seeds)",
     "MAE": seeds["test MAE"].mean() if TORCH_AVAILABLE else np.nan,
     "Source": "this notebook"},
    {"Model": "Ridge", "MAE": ridge_mae, "Source": "this notebook"},
    {"Model": "SARIMAX (B02)", "MAE": 413.8, "Source": "Notebook B02"},
]).sort_values("MAE").reset_index(drop=True)

comparison.round(1)

In [ ]:
if TORCH_AVAILABLE:
    fig, ax = plt.subplots(figsize=(13, 4.5))

    ax.plot(y_test.index, y_test.values, color="black", linewidth=1.6, label="Actual")
    ax.plot(y_test.index, mlp_prediction, color="steelblue", linewidth=1.3,
            linestyle="--", label="MLP (seed 0)")

    ax.set_title("The network's forecast over the test period", fontsize=13, fontweight="bold")
    ax.set_xlabel("Date")
    ax.set_ylabel("Sales")
    ax.legend(loc="upper left")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

The MLP sits **between the linear model and the tree ensembles**. It clearly beats Ridge, which is the
result the theory predicts: the network finds non-linear structure that a weighted sum cannot represent.
And it clearly loses to the random forest and LightGBM.

That ordering is not a failure of deep learning, it is a statement about the dataset. **824 training
rows** is a small number for a model with a few thousand parameters. Gradient boosting was designed to
work well on exactly this scale of tabular data, and it does. Neural networks start to win when there is
much more data, when several series can be learned jointly, or when the input is raw sequence rather than
a hand-built feature matrix, which is where the next three notebooks go.

The plot shows the forecast tracking the weekly rhythm and the closures correctly, which is worth noting
given that nobody told the network what a week is. It inferred that from `day_of_week` and the lag
columns.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="8.-What-an-MLP-Cannot-Do">8. What an MLP Cannot Do</h3>
</div>

The MLP has one structural limitation, and naming it precisely explains the rest of Part D.

**It has no memory, and no notion of order.** The network sees a row of 20 numbers. That `lag_1` comes
immediately before the target, and `lag_2` before that, is something *we* encoded by choosing those
columns; to the network they are just inputs 1 and 2, no more related to each other than `promo` is to
`month`. Shuffle the feature columns and retrain, and the model is identical.

Three consequences follow:

**The window is fixed in advance.** We chose lags up to 28. Anything older is invisible, and extending the
window means more parameters and more rows lost at the start.

**Feature engineering is still doing the work.** The promise of deep learning is that the model learns
features from raw data. This one learned nothing of the sort: it was handed the same matrix as LightGBM
and did slightly worse with it.

**Patterns must be re-learned at every position.** Because there is no sharing across time, the network
learns the relationship between `lag_1` and the target separately from the relationship between `lag_7`
and the target, when a model that understood sequence could learn one rule and apply it at each step.

Those three are exactly what the next notebooks address:

| Notebook | What it adds |
|---|---|
| [D02](./D02_Recurrent_networks.ipynb) | Recurrent networks: a hidden state carried along the sequence |
| [D03](./D03_Convolutional_networks.ipynb) | Convolutions: one filter applied at every position |
| [D04](./D04_Transformers.ipynb) | Attention: every step can look at every other directly |

All of them read the sequence itself rather than a table built from it.

**Exercise.** Retrain the MLP with only the calendar and known-in-advance features, dropping every lag and rolling column. How much worse does it get, and is the difference larger than the seed-to-seed spread measured above? The comparison is the value of the history the network cannot see for itself.

In [ ]:
# Your solution here


---

You now have a network trained on a forecasting problem, and a clear-eyed view of what it did and did not
buy. The next notebook gives the model an actual sense of sequence:
[D02 - Recurrent Neural Networks](./D02_Recurrent_networks.ipynb).